In [1]:
import sys

In [2]:
sys.path.append('/data02/tabm')

In [3]:
# ruff: noqa: E402
import math
import random
import warnings
from typing import Literal, NamedTuple

import numpy as np
import pandas as pd
import rtdl_num_embeddings  # https://github.com/yandex-research/rtdl-num-embeddings
import scipy.special
import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import torch
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from tqdm.std import tqdm

warnings.simplefilter('ignore')
from tabm_reference import Model, make_parameter_groups

warnings.resetwarnings()

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

In [ ]:
import torchvision

In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

NameError: name 'torch' is not defined

In [16]:
df_cb = pd.read_csv('chem_bert_emb_df.csv')
df_cb

,0,2,3,4,5,6,7,8,9,10,...,590,591,592,593,594,595,596,597,598,599
0,0.383230,0.517380,-0.280701,0.008003,0.219609,0.726157,-0.362477,-0.405935,0.430935,-0.859224,...,0.088762,-0.032460,0.060221,0.710166,0.107282,-0.070665,-0.364548,0.524827,-0.219044,0.296993
1,0.053002,0.046581,-0.045467,0.265602,0.121656,0.248467,-0.471550,-0.289988,-0.205801,-0.737166,...,0.241539,0.247456,-0.045622,0.724922,0.419859,-0.171350,-0.007923,0.040881,0.082113,0.109209
2,0.650106,1.011580,0.463880,-0.496643,0.410833,0.831479,0.182609,-0.388742,-0.045355,-0.275201,...,-0.128667,-0.135553,0.645528,-0.111434,0.043732,0.342703,0.008101,0.571934,0.079836,-0.283608
3,-0.209592,-0.243905,0.176476,0.517554,0.026938,-0.402658,-0.327484,-0.280088,-0.028855,-0.180202,...,-0.035334,-0.285914,-0.212273,0.117536,0.522993,-0.621208,0.539129,0.111357,0.331581,-0.086567
4,0.122910,-0.064467,0.686867,0.239316,-0.103160,-0.167726,-0.105149,-0.489707,-0.066420,-0.162347,...,-0.030928,-0.134271,-0.102144,0.248343,0.010503,0.611926,0.318152,-0.114292,0.528896,0.231508
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.217291,0.508175,-0.111223,0.113894,0.382003,0.612502,-0.328363,-0.080576,-0.240976,0.293184,...,-0.045533,0.423016,0.121292,-0.209866,0.408977,0.190961,-0.239830,0.351643,0.320553,-0.202831
117748,-0.067462,0.677433,-0.245434,0.440785,0.263820,0.551249,-0.395648,-0.091693,-0.263357,-0.179503,...,0.316211,0.689446,0.232293,-0.200617,0.187333,0.386570,-0.406295,0.402920,0.360022,-0.122600
117749,0.181496,0.512077,-0.280476,0.437974,-0.027109,0.723539,-0.385805,-0.248053,-0.145493,0.086625,...,-0.096264,0.213210,0.422839,0.372862,0.592165,0.328713,-0.316499,0.294498,0.418610,-0.106374
117750,-0.098858,0.585181,-0.392476,0.777918,0.295732,0.250359,-0.306643,0.118758,-0.226906,-0.079321,...,0.182980,0.544464,0.256705,0.045325,0.412411,0.184333,-0.314178,-0.058447,0.660162,-0.180543


In [17]:
df = pd.read_csv('df.csv')
mps = [mp for mp in df['MP']]
df_cb['MP'] = mps
df_cb

,0,2,3,4,5,6,7,8,9,10,...,591,592,593,594,595,596,597,598,599,MP
0,0.383230,0.517380,-0.280701,0.008003,0.219609,0.726157,-0.362477,-0.405935,0.430935,-0.859224,...,-0.032460,0.060221,0.710166,0.107282,-0.070665,-0.364548,0.524827,-0.219044,0.296993,355.15
1,0.053002,0.046581,-0.045467,0.265602,0.121656,0.248467,-0.471550,-0.289988,-0.205801,-0.737166,...,0.247456,-0.045622,0.724922,0.419859,-0.171350,-0.007923,0.040881,0.082113,0.109209,373.65
2,0.650106,1.011580,0.463880,-0.496643,0.410833,0.831479,0.182609,-0.388742,-0.045355,-0.275201,...,-0.135553,0.645528,-0.111434,0.043732,0.342703,0.008101,0.571934,0.079836,-0.283608,373.75
3,-0.209592,-0.243905,0.176476,0.517554,0.026938,-0.402658,-0.327484,-0.280088,-0.028855,-0.180202,...,-0.285914,-0.212273,0.117536,0.522993,-0.621208,0.539129,0.111357,0.331581,-0.086567,373.70
4,0.122910,-0.064467,0.686867,0.239316,-0.103160,-0.167726,-0.105149,-0.489707,-0.066420,-0.162347,...,-0.134271,-0.102144,0.248343,0.010503,0.611926,0.318152,-0.114292,0.528896,0.231508,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.217291,0.508175,-0.111223,0.113894,0.382003,0.612502,-0.328363,-0.080576,-0.240976,0.293184,...,0.423016,0.121292,-0.209866,0.408977,0.190961,-0.239830,0.351643,0.320553,-0.202831,395.65
117748,-0.067462,0.677433,-0.245434,0.440785,0.263820,0.551249,-0.395648,-0.091693,-0.263357,-0.179503,...,0.689446,0.232293,-0.200617,0.187333,0.386570,-0.406295,0.402920,0.360022,-0.122600,396.15
117749,0.181496,0.512077,-0.280476,0.437974,-0.027109,0.723539,-0.385805,-0.248053,-0.145493,0.086625,...,0.213210,0.422839,0.372862,0.592165,0.328713,-0.316499,0.294498,0.418610,-0.106374,327.65
117750,-0.098858,0.585181,-0.392476,0.777918,0.295732,0.250359,-0.306643,0.118758,-0.226906,-0.079321,...,0.544464,0.256705,0.045325,0.412411,0.184333,-0.314178,-0.058447,0.660162,-0.180543,488.15


In [18]:
res_df = df_cb

res_df

,0,2,3,4,5,6,7,8,9,10,...,591,592,593,594,595,596,597,598,599,MP
0,0.383230,0.517380,-0.280701,0.008003,0.219609,0.726157,-0.362477,-0.405935,0.430935,-0.859224,...,-0.032460,0.060221,0.710166,0.107282,-0.070665,-0.364548,0.524827,-0.219044,0.296993,355.15
1,0.053002,0.046581,-0.045467,0.265602,0.121656,0.248467,-0.471550,-0.289988,-0.205801,-0.737166,...,0.247456,-0.045622,0.724922,0.419859,-0.171350,-0.007923,0.040881,0.082113,0.109209,373.65
2,0.650106,1.011580,0.463880,-0.496643,0.410833,0.831479,0.182609,-0.388742,-0.045355,-0.275201,...,-0.135553,0.645528,-0.111434,0.043732,0.342703,0.008101,0.571934,0.079836,-0.283608,373.75
3,-0.209592,-0.243905,0.176476,0.517554,0.026938,-0.402658,-0.327484,-0.280088,-0.028855,-0.180202,...,-0.285914,-0.212273,0.117536,0.522993,-0.621208,0.539129,0.111357,0.331581,-0.086567,373.70
4,0.122910,-0.064467,0.686867,0.239316,-0.103160,-0.167726,-0.105149,-0.489707,-0.066420,-0.162347,...,-0.134271,-0.102144,0.248343,0.010503,0.611926,0.318152,-0.114292,0.528896,0.231508,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.217291,0.508175,-0.111223,0.113894,0.382003,0.612502,-0.328363,-0.080576,-0.240976,0.293184,...,0.423016,0.121292,-0.209866,0.408977,0.190961,-0.239830,0.351643,0.320553,-0.202831,395.65
117748,-0.067462,0.677433,-0.245434,0.440785,0.263820,0.551249,-0.395648,-0.091693,-0.263357,-0.179503,...,0.689446,0.232293,-0.200617,0.187333,0.386570,-0.406295,0.402920,0.360022,-0.122600,396.15
117749,0.181496,0.512077,-0.280476,0.437974,-0.027109,0.723539,-0.385805,-0.248053,-0.145493,0.086625,...,0.213210,0.422839,0.372862,0.592165,0.328713,-0.316499,0.294498,0.418610,-0.106374,327.65
117750,-0.098858,0.585181,-0.392476,0.777918,0.295732,0.250359,-0.306643,0.118758,-0.226906,-0.079321,...,0.544464,0.256705,0.045325,0.412411,0.184333,-0.314178,-0.058447,0.660162,-0.180543,488.15


In [20]:
res_df.to_csv('chem_bert_emb_df.csv', index=False)

In [ ]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = res_df
X_cont: np.ndarray = res_df.iloc[:, :-1].to_numpy()
Y: np.ndarray = res_df.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

In [42]:
print('\n\nResult:')
print(best)



Result:
{'val': -36.008947320206055, 'test': -36.450139518230074, 'epoch': 153}


In [26]:
df_um = pd.read_csv('uni-mol.csv')
df_um

,0,1,2,3,4,5,6,7,8,9,...,503,504,505,506,507,508,509,510,511,MP
0,-0.013992,-0.669688,-0.810371,-0.820527,-0.944870,-1.545177,1.004045,0.720911,-0.538595,1.131984,...,0.365982,1.061863,-2.759452,1.193249,-0.218466,0.379487,2.335730,0.893357,-2.023531,355.15
1,-0.255941,-0.482257,-0.014340,-0.300146,-0.454813,-1.467290,0.742272,0.779284,-0.378655,1.614774,...,0.528400,0.815105,-2.774471,1.246665,0.295793,-0.397826,2.357099,0.496361,-2.069937,373.65
2,-0.116311,-0.716807,-0.596487,-0.134767,-0.437958,-1.901230,1.060897,0.929643,-0.660413,1.297815,...,0.430205,0.736438,-2.816033,1.252844,0.396341,-0.504774,2.235858,0.548843,-2.080271,373.75
3,-0.094511,-0.824312,-0.569022,-0.392049,-0.464151,-1.441643,0.799341,0.841834,-0.314347,1.312179,...,0.576383,0.337680,-2.739552,1.275348,0.565805,-0.853423,2.139378,0.804444,-2.055217,373.70
4,-0.291880,-0.746322,-0.622199,-0.409220,-0.443025,-1.259129,0.990855,0.646626,-0.346968,1.213809,...,0.606900,0.408470,-2.747625,1.222448,0.677137,-0.627984,2.115620,0.872131,-2.057241,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,-0.073738,-0.700036,-0.405322,-0.244470,-0.489630,-1.644408,0.836901,0.682566,-0.335481,1.285728,...,0.198654,0.263748,-2.825043,1.211132,0.321607,-0.647005,2.305871,0.891368,-2.107719,395.65
117748,-0.146027,-0.864437,-0.300155,-0.259133,-0.658427,-1.554542,0.857640,0.988294,-0.525199,1.286521,...,0.699391,0.257819,-2.783286,1.274370,0.355458,-0.614098,2.207493,0.802195,-2.024621,396.15
117749,0.096798,-1.041117,-0.263131,-0.502725,-0.931887,-1.409036,0.755577,0.720619,-0.723526,1.525824,...,0.449580,1.015750,-2.798421,1.329739,0.094653,-0.600728,2.345972,0.610362,-2.006141,327.65
117750,-0.367869,-0.537741,-0.650870,-0.120295,-0.444859,-1.763567,1.131350,0.718563,-0.293059,1.283509,...,0.286710,0.295653,-2.660894,1.250393,0.357552,-0.294115,2.211480,0.837793,-2.026178,488.15


In [27]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = df_um
X_cont: np.ndarray = df_um.iloc[:, :-1].to_numpy()
Y: np.ndarray = df_um.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

Device:        CUDA
AMP:           False (dtype: torch.bfloat16)
torch.compile: False


In [28]:
# Choose one of the two configurations below.

# TabM
arch_type = 'tabm'
bins = None

model = Model(
    n_num_features=n_cont_features,
    cat_cardinalities=[], # zero categorical data
    n_classes=n_classes,
    backbone={
        'type': 'MLP',
        'n_blocks': 3 if bins is None else 2,
        'd_block': 512,
        'dropout': 0.1,
    },
    bins=bins,
    num_embeddings=(
        None
        if bins is None
        else {
            'type': 'PiecewiseLinearEmbeddings',
            'd_embedding': 16,
            'activation': False,
            'version': 'B',
        }
    ),
    arch_type=arch_type,
    k=32,
    share_training_batches=True,
).to(device)
optimizer = torch.optim.AdamW(make_parameter_groups(model), lr=2e-3, weight_decay=3e-4)

if compile_model:
    # NOTE
    # `torch.compile` is intentionally called without the `mode` argument
    # (mode="reduce-overhead" caused issues during training with torch==2.0.1).
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode
    
@torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: Tensor) -> Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx] if 'x_cat' in data[part] else None,
        )
        .squeeze(-1)  # Remove the last dimension for regression tasks.
        .float()
    )


base_loss_fn = F.mse_loss if task_type == 'regression' else F.cross_entropy


def loss_fn(y_pred: Tensor, y_true: Tensor) -> Tensor:
    # TabM produces k predictions. Each of them must be trained separately.
    # (regression)     y_pred.shape == (batch_size, k)
    # (classification) y_pred.shape == (batch_size, k, n_classes)
    k = y_pred.shape[-1 if task_type == 'regression' else -2]
    return base_loss_fn(
        y_pred.flatten(0, 1),
        y_true.repeat_interleave(k) if model.share_training_batches else y_true,
    )


@evaluation_mode()
def evaluate(part: str) -> float:
    model.eval()

    # When using torch.compile, you may need to reduce the evaluation batch size.
    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    if task_type == 'regression':
        # Transform the predictions back to the original label space.
        assert regression_label_stats is not None
        y_pred = y_pred * regression_label_stats.std + regression_label_stats.mean

    # Compute the mean of the k predictions.
    if task_type != 'regression':
        # For classification, the mean must be computed in the probabily space.
        y_pred = scipy.special.softmax(y_pred, axis=-1)
    y_pred = y_pred.mean(1)

    y_true = data[part]['y'].cpu().numpy()
    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    print(f'MAE ({part} set): {mae:.4f}')  # [ADDED MAE PRINT]
    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
        if task_type == 'regression'
        else sklearn.metrics.accuracy_score(y_true, y_pred.argmax(1))
    )
    return float(score)  # The higher -- the better.


print(f'Test score before training: {evaluate("test"):.4f}')

MAE (test set): 55.9930
Test score before training: -71.3111


In [29]:
%time
n_epochs = 1_000_000_000
patience = 16

train_size = len(train_idx)
batch_size = 256
epoch_size = math.ceil(train_size / batch_size)
best = {
    'val': -math.inf,
    'test': -math.inf,
    'epoch': -1,
}
# Early stopping: the training stops when
# there are more than `patience` consequtive bad updates.
patience = 16
remaining_patience = patience

print('-' * 88 + '\n')
for epoch in range(n_epochs):
    batches = (
        torch.randperm(train_size, device=device).split(batch_size)
        if model.share_training_batches
        else [
            x.transpose(0, 1).flatten()
            for x in torch.rand((model.k, train_size), device=device)
            .argsort(dim=1)
            .split(batch_size, dim=1)
        ]
    )
    for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
        if grad_scaler is None:
            loss.backward()
            optimizer.step()
        else:
            grad_scaler.scale(loss).backward()  # type: ignore
            grad_scaler.step(optimizer)
            grad_scaler.update()

    val_score = evaluate('val')
    test_score = evaluate('test')
    print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

    if val_score > best['val']:
        print('🌸 New best epoch! 🌸')
        best = {'val': val_score, 'test': test_score, 'epoch': epoch}
        remaining_patience = patience
    else:
        remaining_patience -= 1

    if remaining_patience < 0:
        break

    print()

print('\n\nResult:')
print(best)

CPU times: user 7 µs, sys: 0 ns, total: 7 µs
Wall time: 13.8 µs
----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 295/295 [00:03<00:00, 82.46it/s]


MAE (val set): 33.6626
MAE (test set): 33.3756
(val) -43.2836 (test) -43.0816
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 295/295 [00:02<00:00, 102.66it/s]


MAE (val set): 33.1481
MAE (test set): 33.0004
(val) -42.4757 (test) -42.3713
🌸 New best epoch! 🌸



...

MAE (val set): 29.6335
MAE (test set): 29.5816
(val) -38.5560 (test) -38.4735



Epoch 79: 100%|██████████| 295/295 [00:03<00:00, 89.74it/s] 


MAE (val set): 29.4518
MAE (test set): 29.3446
(val) -38.5001 (test) -38.3605



Epoch 80: 100%|██████████| 295/295 [00:02<00:00, 135.01it/s]


MAE (val set): 29.5776
MAE (test set): 29.5541
(val) -38.6487 (test) -38.5216


Result:
{'val': -38.39060592185118, 'test': -38.329470586347625, 'epoch': 63}


In [30]:
print('\n\nResult:')
print(best)



Result:
{'val': -38.39060592185118, 'test': -38.329470586347625, 'epoch': 63}


In [32]:
df_mf = pd.read_csv('molformer-emb.csv')
df_mf

,0,1,2,3,4,5,6,7,8,9,...,759,760,761,762,763,764,765,766,767,MP
0,0.232754,0.039662,0.603345,0.434351,-0.235619,-0.008117,-0.240256,-0.327148,-0.622383,-0.242045,...,0.511787,0.017956,-0.102726,0.593849,-0.322905,-2.748505,0.264649,-0.493542,0.170734,355.15
1,-0.035469,0.244099,0.824150,0.871007,0.381988,0.018918,-0.171494,-0.681534,0.314292,0.073357,...,-0.444052,-0.211882,-0.301885,-0.089473,-0.258444,-2.224507,0.016750,-0.915583,0.149851,373.65
2,-0.014357,0.257718,0.659238,0.541999,0.342051,0.048318,0.188520,-0.241498,-0.025705,-0.207805,...,-0.124264,0.174638,-0.048913,-0.015672,-0.385274,-2.904102,0.212661,-0.588502,-0.315535,373.75
3,0.392396,0.213285,0.820162,0.444167,-0.283028,-0.213827,0.098154,-0.381828,0.016981,-0.291995,...,-0.253576,-0.132808,0.243490,0.575032,0.081913,-2.777802,0.065183,-0.286023,-0.647094,373.70
4,0.630157,0.129021,0.930085,0.348113,-0.079924,-0.057398,0.470295,0.097515,-0.180795,-0.171304,...,0.135372,-0.524551,0.513307,-0.236967,-0.219924,-3.077367,0.617281,-0.337038,-0.176350,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.536852,0.122638,0.304402,-0.319562,-0.675866,0.010863,-0.525479,-0.008692,-0.520580,-0.474743,...,-0.322050,-0.702077,0.462894,0.641812,-0.209804,-2.338952,0.601882,-0.391491,-0.380140,395.65
117748,-0.091166,0.245632,-0.179062,0.299272,-0.800757,-0.122798,-0.938942,-0.098464,0.014622,-0.054302,...,-0.299069,0.062411,0.707462,0.116971,-0.379527,-2.399415,0.433567,-0.402556,-0.216447,396.15
117749,0.463926,0.466987,0.087009,0.034810,-1.054933,-0.277966,-0.886445,-0.060067,-0.231398,0.083403,...,0.006923,-0.156032,0.417531,0.423048,-0.546653,-2.341095,0.170920,-0.561870,-0.619896,327.65
117750,0.502144,0.065869,0.162445,-0.160113,-1.070973,-0.249068,-0.573376,-0.082146,0.197232,-0.234873,...,-0.269982,-0.340257,0.310560,0.339327,-0.547660,-2.511800,0.631499,-0.330420,-0.576203,488.15


In [33]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = df_mf
X_cont: np.ndarray = df_mf.iloc[:, :-1].to_numpy()
Y: np.ndarray = df_mf.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

Device:        CUDA
AMP:           False (dtype: torch.bfloat16)
torch.compile: False


In [34]:
# Choose one of the two configurations below.

# TabM
arch_type = 'tabm'
bins = None

model = Model(
    n_num_features=n_cont_features,
    cat_cardinalities=[], # zero categorical data
    n_classes=n_classes,
    backbone={
        'type': 'MLP',
        'n_blocks': 3 if bins is None else 2,
        'd_block': 512,
        'dropout': 0.1,
    },
    bins=bins,
    num_embeddings=(
        None
        if bins is None
        else {
            'type': 'PiecewiseLinearEmbeddings',
            'd_embedding': 16,
            'activation': False,
            'version': 'B',
        }
    ),
    arch_type=arch_type,
    k=32,
    share_training_batches=True,
).to(device)
optimizer = torch.optim.AdamW(make_parameter_groups(model), lr=2e-3, weight_decay=3e-4)

if compile_model:
    # NOTE
    # `torch.compile` is intentionally called without the `mode` argument
    # (mode="reduce-overhead" caused issues during training with torch==2.0.1).
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode
    
@torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: Tensor) -> Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx] if 'x_cat' in data[part] else None,
        )
        .squeeze(-1)  # Remove the last dimension for regression tasks.
        .float()
    )


base_loss_fn = F.mse_loss if task_type == 'regression' else F.cross_entropy


def loss_fn(y_pred: Tensor, y_true: Tensor) -> Tensor:
    # TabM produces k predictions. Each of them must be trained separately.
    # (regression)     y_pred.shape == (batch_size, k)
    # (classification) y_pred.shape == (batch_size, k, n_classes)
    k = y_pred.shape[-1 if task_type == 'regression' else -2]
    return base_loss_fn(
        y_pred.flatten(0, 1),
        y_true.repeat_interleave(k) if model.share_training_batches else y_true,
    )


@evaluation_mode()
def evaluate(part: str) -> float:
    model.eval()

    # When using torch.compile, you may need to reduce the evaluation batch size.
    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    if task_type == 'regression':
        # Transform the predictions back to the original label space.
        assert regression_label_stats is not None
        y_pred = y_pred * regression_label_stats.std + regression_label_stats.mean

    # Compute the mean of the k predictions.
    if task_type != 'regression':
        # For classification, the mean must be computed in the probabily space.
        y_pred = scipy.special.softmax(y_pred, axis=-1)
    y_pred = y_pred.mean(1)

    y_true = data[part]['y'].cpu().numpy()
    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    print(f'MAE ({part} set): {mae:.4f}')  # [ADDED MAE PRINT]
    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
        if task_type == 'regression'
        else sklearn.metrics.accuracy_score(y_true, y_pred.argmax(1))
    )
    return float(score)  # The higher -- the better.


print(f'Test score before training: {evaluate("test"):.4f}')

MAE (test set): 56.2370
Test score before training: -71.5915


In [35]:
%time
n_epochs = 1_000_000_000
patience = 16

train_size = len(train_idx)
batch_size = 256
epoch_size = math.ceil(train_size / batch_size)
best = {
    'val': -math.inf,
    'test': -math.inf,
    'epoch': -1,
}
# Early stopping: the training stops when
# there are more than `patience` consequtive bad updates.
patience = 16
remaining_patience = patience

print('-' * 88 + '\n')
for epoch in range(n_epochs):
    batches = (
        torch.randperm(train_size, device=device).split(batch_size)
        if model.share_training_batches
        else [
            x.transpose(0, 1).flatten()
            for x in torch.rand((model.k, train_size), device=device)
            .argsort(dim=1)
            .split(batch_size, dim=1)
        ]
    )
    for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
        if grad_scaler is None:
            loss.backward()
            optimizer.step()
        else:
            grad_scaler.scale(loss).backward()  # type: ignore
            grad_scaler.step(optimizer)
            grad_scaler.update()

    val_score = evaluate('val')
    test_score = evaluate('test')
    print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

    if val_score > best['val']:
        print('🌸 New best epoch! 🌸')
        best = {'val': val_score, 'test': test_score, 'epoch': epoch}
        remaining_patience = patience
    else:
        remaining_patience -= 1

    if remaining_patience < 0:
        break

    print()

print('\n\nResult:')
print(best)

CPU times: user 31 µs, sys: 0 ns, total: 31 µs
Wall time: 64.4 µs
----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 295/295 [00:03<00:00, 82.15it/s] 


MAE (val set): 31.8265
MAE (test set): 32.3269
(val) -41.2789 (test) -41.8286
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 295/295 [00:03<00:00, 91.49it/s] 


MAE (val set): 30.4734
MAE (test set): 30.8124
(val) -40.1065 (test) -40.4765
🌸 New best epoch! 🌸



...

MAE (val set): 27.6307
MAE (test set): 28.0061
(val) -36.5165 (test) -36.9406



Epoch 147: 100%|██████████| 295/295 [00:03<00:00, 88.56it/s] 


MAE (val set): 27.5660
MAE (test set): 27.9449
(val) -36.5416 (test) -36.9478



Epoch 148: 100%|██████████| 295/295 [00:03<00:00, 93.19it/s] 


MAE (val set): 27.6304
MAE (test set): 27.9688
(val) -36.5164 (test) -36.8982


Result:
{'val': -36.44182978402446, 'test': -36.85538077129139, 'epoch': 131}


In [37]:
print('\n\nResult:')
print(best)



Result:
{'val': -36.44182978402446, 'test': -36.85538077129139, 'epoch': 131}


In [46]:
df_mr = pd.read_csv('emb_morgan_df.csv')
df_mr

,0,1,2,3,4,5,6,7,8,9,...,4087,4088,4089,4090,4091,4092,4093,4094,4095,MP
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,355.15
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,373.65
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,373.75
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,373.70
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,395.65
117748,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,396.15
117749,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,327.65
117750,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,488.15


In [47]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = df_mr
X_cont: np.ndarray = df_mr.iloc[:, :-1].to_numpy()
Y: np.ndarray = df_mr.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

Device:        CUDA
AMP:           False (dtype: torch.bfloat16)
torch.compile: False


In [55]:
# Choose one of the two configurations below.

# TabM
arch_type = 'tabm'
bins = None

model = Model(
    n_num_features=n_cont_features,
    cat_cardinalities=[], # zero categorical data
    n_classes=n_classes,
    backbone={
        'type': 'MLP',
        'n_blocks': 3 if bins is None else 2,
        'd_block': 512,
        'dropout': 0.1,
    },
    bins=bins,
    num_embeddings=(
        None
        if bins is None
        else {
            'type': 'PiecewiseLinearEmbeddings',
            'd_embedding': 16,
            'activation': False,
            'version': 'B',
        }
    ),
    arch_type=arch_type,
    k=32,
    share_training_batches=True,
).to(device)
optimizer = torch.optim.AdamW(make_parameter_groups(model), lr=2e-3, weight_decay=3e-4)

if compile_model:
    # NOTE
    # `torch.compile` is intentionally called without the `mode` argument
    # (mode="reduce-overhead" caused issues during training with torch==2.0.1).
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode
    
@torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: Tensor) -> Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx] if 'x_cat' in data[part] else None,
        )
        .squeeze(-1)  # Remove the last dimension for regression tasks.
        .float()
    )


base_loss_fn = F.mse_loss if task_type == 'regression' else F.cross_entropy


def loss_fn(y_pred: Tensor, y_true: Tensor) -> Tensor:
    # TabM produces k predictions. Each of them must be trained separately.
    # (regression)     y_pred.shape == (batch_size, k)
    # (classification) y_pred.shape == (batch_size, k, n_classes)
    k = y_pred.shape[-1 if task_type == 'regression' else -2]
    return base_loss_fn(
        y_pred.flatten(0, 1),
        y_true.repeat_interleave(k) if model.share_training_batches else y_true,
    )


@evaluation_mode()
def evaluate(part: str) -> float:
    import time  # Import time module for timing
    start_time = time.time()  # Start timer
    
    model.eval()

    # When using torch.compile, you may need to reduce the evaluation batch size.
    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    if task_type == 'regression':
        # Transform the predictions back to the original label space.
        assert regression_label_stats is not None
        y_pred = y_pred * regression_label_stats.std + regression_label_stats.mean

    # Compute the mean of the k predictions.
    if task_type != 'regression':
        # For classification, the mean must be computed in the probabily space.
        y_pred = scipy.special.softmax(y_pred, axis=-1)
    y_pred = y_pred.mean(1)

    y_true = data[part]['y'].cpu().numpy()
    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    print(f'MAE ({part} set): {mae:.4f}')  # [ADDED MAE PRINT]
    
    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
        if task_type == 'regression'
        else sklearn.metrics.accuracy_score(y_true, y_pred.argmax(1))
    )
    
    end_time = time.time()  # End timer
    elapsed_time = end_time - start_time  # Calculate elapsed time
    print(f'Evaluation completed in {elapsed_time:.2f} seconds')  # Print timing info
    
    return float(score)  # The higher -- the better.


print(f'Test score before training: {evaluate("test"):.4f}')

MAE (test set): 56.0152
Evaluation completed in 0.53 seconds
Test score before training: -71.2805


In [57]:
n_epochs = 1_000_000_000
patience = 16

train_size = len(train_idx)
batch_size = 256
epoch_size = math.ceil(train_size / batch_size)
best = {
    'val': -math.inf,
    'test': -math.inf,
    'epoch': -1,
}
# Early stopping: the training stops when
# there are more than `patience` consequtive bad updates.
patience = 16
remaining_patience = patience

print('-' * 88 + '\n')
for epoch in range(n_epochs):
    batches = (
        torch.randperm(train_size, device=device).split(batch_size)
        if model.share_training_batches
        else [
            x.transpose(0, 1).flatten()
            for x in torch.rand((model.k, train_size), device=device)
            .argsort(dim=1)
            .split(batch_size, dim=1)
        ]
    )
    for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
        if grad_scaler is None:
            loss.backward()
            optimizer.step()
        else:
            grad_scaler.scale(loss).backward()  # type: ignore
            grad_scaler.step(optimizer)
            grad_scaler.update()

    val_score = evaluate('val')
    test_score = evaluate('test')
    print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

    if val_score > best['val']:
        print('🌸 New best epoch! 🌸')
        best = {'val': val_score, 'test': test_score, 'epoch': epoch}
        remaining_patience = patience
    else:
        remaining_patience -= 1

    if remaining_patience < 0:
        break

    print()

print('\n\nResult:')
print(best)

CPU times: user 18 µs, sys: 0 ns, total: 18 µs
Wall time: 75.8 µs
----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 295/295 [00:05<00:00, 55.98it/s]


MAE (val set): 32.7708
Evaluation completed in 0.64 seconds
MAE (test set): 32.7285
Evaluation completed in 0.53 seconds
(val) -42.6536 (test) -42.7311
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 295/295 [00:05<00:00, 56.75it/s]


MAE (val set): 30.6051
Evaluation completed in 0.67 seconds
MAE (test set): 30.7552
Evaluation completed in 0.53 seconds
(val) -40.1447 (test) -40.3202
🌸 New best epoch! 🌸



...

MAE (val set): 27.6264
Evaluation completed in 0.63 seconds
MAE (test set): 27.7320
Evaluation completed in 0.54 seconds
(val) -37.0290 (test) -37.2300



Epoch 55: 100%|██████████| 295/295 [00:05<00:00, 57.70it/s]


MAE (val set): 27.4523
Evaluation completed in 0.63 seconds
MAE (test set): 27.5992
Evaluation completed in 0.52 seconds
(val) -37.0255 (test) -37.2404



Epoch 56: 100%|██████████| 295/295 [00:05<00:00, 57.73it/s]


MAE (val set): 27.7055
Evaluation completed in 0.63 seconds
MAE (test set): 27.8388
Evaluation completed in 0.53 seconds
(val) -37.0619 (test) -37.2726


Result:
{'val': -36.92161711716965, 'test': -37.12807259223218, 'epoch': 39}


In [58]:
print('\n\nResult:')
print(best)



Result:
{'val': -36.92161711716965, 'test': -37.12807259223218, 'epoch': 39}


In [59]:
df_cb

,0,2,3,4,5,6,7,8,9,10,...,591,592,593,594,595,596,597,598,599,MP
0,0.383230,0.517380,-0.280701,0.008003,0.219609,0.726157,-0.362477,-0.405935,0.430935,-0.859224,...,-0.032460,0.060221,0.710166,0.107282,-0.070665,-0.364548,0.524827,-0.219044,0.296993,355.15
1,0.053002,0.046581,-0.045467,0.265602,0.121656,0.248467,-0.471550,-0.289988,-0.205801,-0.737166,...,0.247456,-0.045622,0.724922,0.419859,-0.171350,-0.007923,0.040881,0.082113,0.109209,373.65
2,0.650106,1.011580,0.463880,-0.496643,0.410833,0.831479,0.182609,-0.388742,-0.045355,-0.275201,...,-0.135553,0.645528,-0.111434,0.043732,0.342703,0.008101,0.571934,0.079836,-0.283608,373.75
3,-0.209592,-0.243905,0.176476,0.517554,0.026938,-0.402658,-0.327484,-0.280088,-0.028855,-0.180202,...,-0.285914,-0.212273,0.117536,0.522993,-0.621208,0.539129,0.111357,0.331581,-0.086567,373.70
4,0.122910,-0.064467,0.686867,0.239316,-0.103160,-0.167726,-0.105149,-0.489707,-0.066420,-0.162347,...,-0.134271,-0.102144,0.248343,0.010503,0.611926,0.318152,-0.114292,0.528896,0.231508,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.217291,0.508175,-0.111223,0.113894,0.382003,0.612502,-0.328363,-0.080576,-0.240976,0.293184,...,0.423016,0.121292,-0.209866,0.408977,0.190961,-0.239830,0.351643,0.320553,-0.202831,395.65
117748,-0.067462,0.677433,-0.245434,0.440785,0.263820,0.551249,-0.395648,-0.091693,-0.263357,-0.179503,...,0.689446,0.232293,-0.200617,0.187333,0.386570,-0.406295,0.402920,0.360022,-0.122600,396.15
117749,0.181496,0.512077,-0.280476,0.437974,-0.027109,0.723539,-0.385805,-0.248053,-0.145493,0.086625,...,0.213210,0.422839,0.372862,0.592165,0.328713,-0.316499,0.294498,0.418610,-0.106374,327.65
117750,-0.098858,0.585181,-0.392476,0.777918,0.295732,0.250359,-0.306643,0.118758,-0.226906,-0.079321,...,0.544464,0.256705,0.045325,0.412411,0.184333,-0.314178,-0.058447,0.660162,-0.180543,488.15


In [60]:
df_mf

,0,1,2,3,4,5,6,7,8,9,...,759,760,761,762,763,764,765,766,767,MP
0,0.232754,0.039662,0.603345,0.434351,-0.235619,-0.008117,-0.240256,-0.327148,-0.622383,-0.242045,...,0.511787,0.017956,-0.102726,0.593849,-0.322905,-2.748505,0.264649,-0.493542,0.170734,355.15
1,-0.035469,0.244099,0.824150,0.871007,0.381988,0.018918,-0.171494,-0.681534,0.314292,0.073357,...,-0.444052,-0.211882,-0.301885,-0.089473,-0.258444,-2.224507,0.016750,-0.915583,0.149851,373.65
2,-0.014357,0.257718,0.659238,0.541999,0.342051,0.048318,0.188520,-0.241498,-0.025705,-0.207805,...,-0.124264,0.174638,-0.048913,-0.015672,-0.385274,-2.904102,0.212661,-0.588502,-0.315535,373.75
3,0.392396,0.213285,0.820162,0.444167,-0.283028,-0.213827,0.098154,-0.381828,0.016981,-0.291995,...,-0.253576,-0.132808,0.243490,0.575032,0.081913,-2.777802,0.065183,-0.286023,-0.647094,373.70
4,0.630157,0.129021,0.930085,0.348113,-0.079924,-0.057398,0.470295,0.097515,-0.180795,-0.171304,...,0.135372,-0.524551,0.513307,-0.236967,-0.219924,-3.077367,0.617281,-0.337038,-0.176350,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.536852,0.122638,0.304402,-0.319562,-0.675866,0.010863,-0.525479,-0.008692,-0.520580,-0.474743,...,-0.322050,-0.702077,0.462894,0.641812,-0.209804,-2.338952,0.601882,-0.391491,-0.380140,395.65
117748,-0.091166,0.245632,-0.179062,0.299272,-0.800757,-0.122798,-0.938942,-0.098464,0.014622,-0.054302,...,-0.299069,0.062411,0.707462,0.116971,-0.379527,-2.399415,0.433567,-0.402556,-0.216447,396.15
117749,0.463926,0.466987,0.087009,0.034810,-1.054933,-0.277966,-0.886445,-0.060067,-0.231398,0.083403,...,0.006923,-0.156032,0.417531,0.423048,-0.546653,-2.341095,0.170920,-0.561870,-0.619896,327.65
117750,0.502144,0.065869,0.162445,-0.160113,-1.070973,-0.249068,-0.573376,-0.082146,0.197232,-0.234873,...,-0.269982,-0.340257,0.310560,0.339327,-0.547660,-2.511800,0.631499,-0.330420,-0.576203,488.15


In [62]:
joined_df = pd.concat([df_cb.drop(columns=['MP']), df_mf], axis=1)
joined_df

,0,2,3,4,5,6,7,8,9,10,...,759,760,761,762,763,764,765,766,767,MP
0,0.383230,0.517380,-0.280701,0.008003,0.219609,0.726157,-0.362477,-0.405935,0.430935,-0.859224,...,0.511787,0.017956,-0.102726,0.593849,-0.322905,-2.748505,0.264649,-0.493542,0.170734,355.15
1,0.053002,0.046581,-0.045467,0.265602,0.121656,0.248467,-0.471550,-0.289988,-0.205801,-0.737166,...,-0.444052,-0.211882,-0.301885,-0.089473,-0.258444,-2.224507,0.016750,-0.915583,0.149851,373.65
2,0.650106,1.011580,0.463880,-0.496643,0.410833,0.831479,0.182609,-0.388742,-0.045355,-0.275201,...,-0.124264,0.174638,-0.048913,-0.015672,-0.385274,-2.904102,0.212661,-0.588502,-0.315535,373.75
3,-0.209592,-0.243905,0.176476,0.517554,0.026938,-0.402658,-0.327484,-0.280088,-0.028855,-0.180202,...,-0.253576,-0.132808,0.243490,0.575032,0.081913,-2.777802,0.065183,-0.286023,-0.647094,373.70
4,0.122910,-0.064467,0.686867,0.239316,-0.103160,-0.167726,-0.105149,-0.489707,-0.066420,-0.162347,...,0.135372,-0.524551,0.513307,-0.236967,-0.219924,-3.077367,0.617281,-0.337038,-0.176350,374.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117747,0.217291,0.508175,-0.111223,0.113894,0.382003,0.612502,-0.328363,-0.080576,-0.240976,0.293184,...,-0.322050,-0.702077,0.462894,0.641812,-0.209804,-2.338952,0.601882,-0.391491,-0.380140,395.65
117748,-0.067462,0.677433,-0.245434,0.440785,0.263820,0.551249,-0.395648,-0.091693,-0.263357,-0.179503,...,-0.299069,0.062411,0.707462,0.116971,-0.379527,-2.399415,0.433567,-0.402556,-0.216447,396.15
117749,0.181496,0.512077,-0.280476,0.437974,-0.027109,0.723539,-0.385805,-0.248053,-0.145493,0.086625,...,0.006923,-0.156032,0.417531,0.423048,-0.546653,-2.341095,0.170920,-0.561870,-0.619896,327.65
117750,-0.098858,0.585181,-0.392476,0.777918,0.295732,0.250359,-0.306643,0.118758,-0.226906,-0.079321,...,-0.269982,-0.340257,0.310560,0.339327,-0.547660,-2.511800,0.631499,-0.330420,-0.576203,488.15


In [63]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = joined_df
X_cont: np.ndarray = joined_df.iloc[:, :-1].to_numpy()
Y: np.ndarray = joined_df.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

Device:        CUDA
AMP:           False (dtype: torch.bfloat16)
torch.compile: False


In [64]:
# Choose one of the two configurations below.

# TabM
arch_type = 'tabm'
bins = None

model = Model(
    n_num_features=n_cont_features,
    cat_cardinalities=[], # zero categorical data
    n_classes=n_classes,
    backbone={
        'type': 'MLP',
        'n_blocks': 3 if bins is None else 2,
        'd_block': 512,
        'dropout': 0.1,
    },
    bins=bins,
    num_embeddings=(
        None
        if bins is None
        else {
            'type': 'PiecewiseLinearEmbeddings',
            'd_embedding': 16,
            'activation': False,
            'version': 'B',
        }
    ),
    arch_type=arch_type,
    k=32,
    share_training_batches=True,
).to(device)
optimizer = torch.optim.AdamW(make_parameter_groups(model), lr=2e-3, weight_decay=3e-4)

if compile_model:
    # NOTE
    # `torch.compile` is intentionally called without the `mode` argument
    # (mode="reduce-overhead" caused issues during training with torch==2.0.1).
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode
    
@torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: Tensor) -> Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx] if 'x_cat' in data[part] else None,
        )
        .squeeze(-1)  # Remove the last dimension for regression tasks.
        .float()
    )


base_loss_fn = F.mse_loss if task_type == 'regression' else F.cross_entropy


def loss_fn(y_pred: Tensor, y_true: Tensor) -> Tensor:
    # TabM produces k predictions. Each of them must be trained separately.
    # (regression)     y_pred.shape == (batch_size, k)
    # (classification) y_pred.shape == (batch_size, k, n_classes)
    k = y_pred.shape[-1 if task_type == 'regression' else -2]
    return base_loss_fn(
        y_pred.flatten(0, 1),
        y_true.repeat_interleave(k) if model.share_training_batches else y_true,
    )


@evaluation_mode()
def evaluate(part: str) -> float:
    import time  # Import time module for timing
    start_time = time.time()  # Start timer
    
    model.eval()

    # When using torch.compile, you may need to reduce the evaluation batch size.
    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    if task_type == 'regression':
        # Transform the predictions back to the original label space.
        assert regression_label_stats is not None
        y_pred = y_pred * regression_label_stats.std + regression_label_stats.mean

    # Compute the mean of the k predictions.
    if task_type != 'regression':
        # For classification, the mean must be computed in the probabily space.
        y_pred = scipy.special.softmax(y_pred, axis=-1)
    y_pred = y_pred.mean(1)

    y_true = data[part]['y'].cpu().numpy()
    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    print(f'MAE ({part} set): {mae:.4f}')  # [ADDED MAE PRINT]
    
    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
        if task_type == 'regression'
        else sklearn.metrics.accuracy_score(y_true, y_pred.argmax(1))
    )
    
    end_time = time.time()  # End timer
    elapsed_time = end_time - start_time  # Calculate elapsed time
    print(f'Evaluation completed in {elapsed_time:.2f} seconds')  # Print timing info
    
    return float(score)  # The higher -- the better.


print(f'Test score before training: {evaluate("test"):.4f}')

MAE (test set): 56.2889
Evaluation completed in 0.29 seconds
Test score before training: -71.3370


In [65]:
n_epochs = 1_000_000_000
patience = 16

train_size = len(train_idx)
batch_size = 256
epoch_size = math.ceil(train_size / batch_size)
best = {
    'val': -math.inf,
    'test': -math.inf,
    'epoch': -1,
}
# Early stopping: the training stops when
# there are more than `patience` consequtive bad updates.
patience = 16
remaining_patience = patience

print('-' * 88 + '\n')
for epoch in range(n_epochs):
    batches = (
        torch.randperm(train_size, device=device).split(batch_size)
        if model.share_training_batches
        else [
            x.transpose(0, 1).flatten()
            for x in torch.rand((model.k, train_size), device=device)
            .argsort(dim=1)
            .split(batch_size, dim=1)
        ]
    )
    for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
        model.train()
        optimizer.zero_grad()
        loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
        if grad_scaler is None:
            loss.backward()
            optimizer.step()
        else:
            grad_scaler.scale(loss).backward()  # type: ignore
            grad_scaler.step(optimizer)
            grad_scaler.update()

    val_score = evaluate('val')
    test_score = evaluate('test')
    print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

    if val_score > best['val']:
        print('🌸 New best epoch! 🌸')
        best = {'val': val_score, 'test': test_score, 'epoch': epoch}
        remaining_patience = patience
    else:
        remaining_patience -= 1

    if remaining_patience < 0:
        break

    print()

print('\n\nResult:')
print(best)

----------------------------------------------------------------------------------------



Epoch 0: 100%|██████████| 295/295 [00:02<00:00, 105.70it/s]


MAE (val set): 30.2410
Evaluation completed in 0.34 seconds
MAE (test set): 30.5868
Evaluation completed in 0.27 seconds
(val) -39.2152 (test) -39.7790
🌸 New best epoch! 🌸



Epoch 1: 100%|██████████| 295/295 [00:02<00:00, 105.41it/s]


MAE (val set): 29.9915
Evaluation completed in 0.34 seconds
MAE (test set): 30.3770
Evaluation completed in 0.28 seconds
(val) -38.8047 (test) -39.3950
🌸 New best epoch! 🌸



...

MAE (val set): 26.3547
Evaluation completed in 0.34 seconds
MAE (test set): 26.8357
Evaluation completed in 0.28 seconds
(val) -34.9058 (test) -35.6491



Epoch 131: 100%|██████████| 295/295 [00:02<00:00, 103.22it/s]


MAE (val set): 26.1555
Evaluation completed in 0.34 seconds
MAE (test set): 26.5773
Evaluation completed in 0.28 seconds
(val) -34.6609 (test) -35.3426



Epoch 132: 100%|██████████| 295/295 [00:02<00:00, 103.31it/s]


MAE (val set): 25.9880
Evaluation completed in 0.34 seconds
MAE (test set): 26.4232
Evaluation completed in 0.28 seconds
(val) -34.5992 (test) -35.2957


Result:
{'val': -34.58191630819314, 'test': -35.27654419458488, 'epoch': 115}


In [66]:
print('\n\nResult:')
print(best)



Result:
{'val': -34.58191630819314, 'test': -35.27654419458488, 'epoch': 115}


In [67]:
# >>> Dataset.
TaskType = Literal['regression', 'binclass', 'multiclass']

# Regression.
task_type: TaskType = 'regression'
n_classes = None
dataset = joined_df
X_cont: np.ndarray = joined_df.iloc[:, :-1].to_numpy()
Y: np.ndarray = joined_df.iloc[:, -1].to_numpy()

task_is_regression = task_type == 'regression'

# >>> Continuous features.
X_cont: np.ndarray = X_cont.astype(np.float32)
n_cont_features = X_cont.shape[1]

# >>> Labels.
if task_type == 'regression':
    Y = Y.astype(np.float32)
else:
    assert n_classes is not None
    Y = Y.astype(np.int64)
    assert set(Y.tolist()) == set(
        range(n_classes)
    ), 'Classification labels must form the range [0, 1, ..., n_classes - 1]'

# >>> Split the dataset.
all_idx = np.arange(len(Y))
trainval_idx, test_idx = sklearn.model_selection.train_test_split(
    all_idx, train_size=0.8
)
train_idx, val_idx = sklearn.model_selection.train_test_split(
    trainval_idx, train_size=0.8
)
data_numpy = {
    'train': {'x_cont': X_cont[train_idx], 'y': Y[train_idx]},
    'val': {'x_cont': X_cont[val_idx], 'y': Y[val_idx]},
    'test': {'x_cont': X_cont[test_idx], 'y': Y[test_idx]},
}

X_cont_train_numpy = data_numpy['train']['x_cont']
noise = (
    np.random.default_rng(0)
    .normal(0.0, 1e-5, X_cont_train_numpy.shape)
    .astype(X_cont_train_numpy.dtype)
)
preprocessing = sklearn.preprocessing.QuantileTransformer(
    n_quantiles=max(min(len(train_idx) // 30, 1000), 10),
    output_distribution='normal',
    subsample=10**9,
).fit(X_cont_train_numpy + noise)
del X_cont_train_numpy

# Apply the preprocessing.
for part in data_numpy:
    data_numpy[part]['x_cont'] = preprocessing.transform(data_numpy[part]['x_cont'])


# Label preprocessing.
class RegressionLabelStats(NamedTuple):
    mean: float
    std: float


Y_train = data_numpy['train']['y'].copy()
if task_type == 'regression':
    # For regression tasks, it is highly recommended to standardize the training labels.
    regression_label_stats = RegressionLabelStats(
        Y_train.mean().item(), Y_train.std().item()
    )
    Y_train = (Y_train - regression_label_stats.mean) / regression_label_stats.std
else:
    regression_label_stats = None

data = {
    part: {k: torch.as_tensor(v, device=device) for k, v in data_numpy[part].items()}
    for part in data_numpy
}

Y_train = torch.as_tensor(Y_train, device=device)
if task_type == 'regression':
    for part in data:
        data[part]['y'] = data[part]['y'].float()
    Y_train = Y_train.float()
    
amp_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
    if torch.cuda.is_available()
    else None
)
# Changing False to True will result in faster training on compatible hardware.
amp_enabled = False and amp_dtype is not None
grad_scaler = torch.cuda.amp.GradScaler() if amp_dtype is torch.float16 else None  # type: ignore

# torch.compile
compile_model = False

# fmt: off
print(
    f'Device:        {device.type.upper()}'
    f'\nAMP:           {amp_enabled} (dtype: {amp_dtype})'
    f'\ntorch.compile: {compile_model}'
)
# fmt: on

Device:        CUDA
AMP:           False (dtype: torch.bfloat16)
torch.compile: False


In [ ]:
# Choose one of the two configurations below.

# TabM
arch_type = 'tabm'
bins = None

model = Model(
    n_num_features=n_cont_features,
    cat_cardinalities=[], # zero categorical data
    n_classes=n_classes,
    backbone={
        'type': 'MLP',
        'n_blocks': 3 if bins is None else 2,
        'd_block': 512,
        'dropout': 0.1,
    },
    bins=bins,
    num_embeddings=(
        None
        if bins is None
        else {
            'type': 'PiecewiseLinearEmbeddings',
            'd_embedding': 16,
            'activation': False,
            'version': 'B',
        }
    ),
    arch_type=arch_type,
    k=32,
    share_training_batches=True,
).to(device)
optimizer = torch.optim.AdamW(make_parameter_groups(model), lr=2e-3, weight_decay=3e-4)

if compile_model:
    # NOTE
    # `torch.compile` is intentionally called without the `mode` argument
    # (mode="reduce-overhead" caused issues during training with torch==2.0.1).
    model = torch.compile(model)
    evaluation_mode = torch.no_grad
else:
    evaluation_mode = torch.inference_mode
    
@torch.autocast(device.type, enabled=amp_enabled, dtype=amp_dtype)  # type: ignore[code]
def apply_model(part: str, idx: Tensor) -> Tensor:
    return (
        model(
            data[part]['x_cont'][idx],
            data[part]['x_cat'][idx] if 'x_cat' in data[part] else None,
        )
        .squeeze(-1)  # Remove the last dimension for regression tasks.
        .float()
    )


base_loss_fn = F.mse_loss if task_type == 'regression' else F.cross_entropy


def loss_fn(y_pred: Tensor, y_true: Tensor) -> Tensor:
    # TabM produces k predictions. Each of them must be trained separately.
    # (regression)     y_pred.shape == (batch_size, k)
    # (classification) y_pred.shape == (batch_size, k, n_classes)
    k = y_pred.shape[-1 if task_type == 'regression' else -2]
    return base_loss_fn(
        y_pred.flatten(0, 1),
        y_true.repeat_interleave(k) if model.share_training_batches else y_true,
    )


@evaluation_mode()
def evaluate(part: str) -> float:
    import time  # Import time module for timing
    start_time = time.time()  # Start timer
    
    model.eval()

    # When using torch.compile, you may need to reduce the evaluation batch size.
    eval_batch_size = 8096
    y_pred: np.ndarray = (
        torch.cat(
            [
                apply_model(part, idx)
                for idx in torch.arange(len(data[part]['y']), device=device).split(
                    eval_batch_size
                )
            ]
        )
        .cpu()
        .numpy()
    )
    if task_type == 'regression':
        # Transform the predictions back to the original label space.
        assert regression_label_stats is not None
        y_pred = y_pred * regression_label_stats.std + regression_label_stats.mean

    # Compute the mean of the k predictions.
    if task_type != 'regression':
        # For classification, the mean must be computed in the probabily space.
        y_pred = scipy.special.softmax(y_pred, axis=-1)
    y_pred = y_pred.mean(1)

    y_true = data[part]['y'].cpu().numpy()
    mae = sklearn.metrics.mean_absolute_error(y_true, y_pred)
    print(f'MAE ({part} set): {mae:.4f}')  # [ADDED MAE PRINT]
    
    score = (
        -(sklearn.metrics.mean_squared_error(y_true, y_pred) ** 0.5)
        if task_type == 'regression'
        else sklearn.metrics.accuracy_score(y_true, y_pred.argmax(1))
    )
    
    end_time = time.time()  # End timer
    elapsed_time = end_time - start_time  # Calculate elapsed time
    print(f'Evaluation completed in {elapsed_time:.2f} seconds')  # Print timing info
    
    return float(score)  # The higher -- the better.


print(f'Test score before training: {evaluate("test"):.4f}')

In [68]:
import optuna
from optuna.samplers import TPESampler

In [70]:
# TabM
arch_type = 'tabm'
bins = None
def train_and_evaluate(model, optimizer):
    n_epochs = 1_000_000_000
    patience = 16

    train_size = len(train_idx)
    batch_size = 256
    epoch_size = math.ceil(train_size / batch_size)
    best = {
        'val': -math.inf,
        'test': -math.inf,
        'epoch': -1,
    }
    # Early stopping: the training stops when
    # there are more than `patience` consequtive bad updates.
    patience = 16
    remaining_patience = patience

    print('-' * 88 + '\n')
    for epoch in range(n_epochs):
        batches = (
            torch.randperm(train_size, device=device).split(batch_size)
            if model.share_training_batches
            else [
                x.transpose(0, 1).flatten()
                for x in torch.rand((model.k, train_size), device=device)
                .argsort(dim=1)
                .split(batch_size, dim=1)
            ]
        )
        for batch_idx in tqdm(batches, desc=f'Epoch {epoch}'):
            model.train()
            optimizer.zero_grad()
            loss = loss_fn(apply_model('train', batch_idx), Y_train[batch_idx])
            if grad_scaler is None:
                loss.backward()
                optimizer.step()
            else:
                grad_scaler.scale(loss).backward()  # type: ignore
                grad_scaler.step(optimizer)
                grad_scaler.update()

        val_score = evaluate('val')
        test_score = evaluate('test')
        print(f'(val) {val_score:.4f} (test) {test_score:.4f}')

        if val_score > best['val']:
            print('🌸 New best epoch! 🌸')
            best = {'val': val_score, 'test': test_score, 'epoch': epoch}
            remaining_patience = patience
        else:
            remaining_patience -= 1

        if remaining_patience < 0:
            break

        print()

    print('\n\nResult:')
    print(best)